## Importy

In [17]:
import numpy as np
import torch
import torch.nn as nn
import math
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import torch.optim as optim
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import time

## Funkcje aktywacji

In [18]:
def sigmoid(z):
    z = np.clip(z, -500, 500)
    return 1 / (1.0 + np.exp(-z))

def linear(z):
    return z

def relu(z):
    return np.maximum(0, z)

def tanh(z):
    return np.tanh(z)

def leaky_relu(x, alpha=0.01):
    return np.where(x >= 0, x, alpha * x)

def softmax(z):
    z_shifted = z - np.max(z, axis=1, keepdims=True) # odejmujemy maksimum w każdym wierszu, żeby uniknąć overflow exp()
    exp_z = np.exp(z_shifted)
    return exp_z / np.sum(exp_z, axis=1, keepdims=True)

## Pochodne funkcji aktywacji

In [19]:
def sigmoid_prime(z):
    s = sigmoid(z)
    return s * (1.0 - s)

def linear_prime(z):
    return np.ones_like(z)

def relu_prime(z):
    return (z > 0).astype(float)

def tanh_prime(z):
    t = np.tanh(z)
    return 1.0 - t**2

def leaky_relu_prime(x, alpha=0.01):
    return np.where(x >= 0, 1.0, alpha)

# definiujemy klasyczną pochodną softmax, ale nie będziemy jej używać w praktyce, ale może kiedyś się przyda 
def softmax_prime(z):
    s = softmax(z)
    m, k = s.shape
    jacobians = np.zeros((m, k, k))
    for i in range(m):
        si = s[i].reshape(-1, 1)
        jacobians[i] = np.diagflat(si) - si @ si.T
    return jacobians

## Metoda do inicjalizacji parametrów

In [20]:
def initiate_parameters(wielkosci_warstw, metoda, seed=None):

    rng = np.random.default_rng(seed)

    wagi = []
    biasy = []

    for i in range(len(wielkosci_warstw) - 1):
        wejscia = wielkosci_warstw[i]
        wyjscia = wielkosci_warstw[i+1]

        if metoda == "uniform01":                             
            W = rng.uniform(-1.0, 1.0, size=(wejscia, wyjscia))     
            b = np.zeros((1, wyjscia))                           
        elif metoda == "xavier":                              
            limit = np.sqrt(6.0 / (wejscia + wyjscia))              
            W = rng.uniform(-limit, limit, size=(wejscia, wyjscia)) 
            b = np.zeros((1, wyjscia))                          
        elif metoda == "he":                                   
            std = np.sqrt(2.0 / wejscia)                        
            W = rng.normal(0.0, std, size=(wejscia, wyjscia))      
            b = np.zeros((1, wyjscia))                           
        else:                                                 
            raise ValueError("Nieznana metoda inicjalizacji.")
        
        wagi.append(W)
        biasy.append(b)
    
    return wagi, biasy

## Funkcja straty i jej pochodna względem wartości przewidywanych

In [21]:
def mse_loss(y_true, y_pred):
    return np.mean((y_pred - y_true) ** 2)

def mse_grad(y_true, y_pred):
    return 2 * (y_pred - y_true) / y_true.size

def cross_entropy_loss(y_true, y_pred, eps=1e-12):
    y_pred = np.clip(y_pred, eps, 1.0 - eps)
    return -np.mean(np.sum(y_true * np.log(y_pred), axis=1))

def cross_entropy_grad_from_softmax(y_true, y_pred):
    m = y_true.shape[0]
    return (y_pred - y_true) / m

## One-hot encoding, macro f1-score i accuracy

In [22]:
def one_hot_encode(y):
    y = np.asarray(y).reshape(-1)
    klasy = np.unique(y)
    mapa = {klasa: i for i, klasa in enumerate(klasy)}
    y_idx = np.array([mapa[klasa] for klasa in y])
    
    one_hot = np.zeros((len(y_idx), len(klasy)))
    one_hot[np.arange(len(y_idx)), y_idx] = 1.0
    
    return one_hot, mapa


def one_hot_encode_with_map(y, mapa):
    y = np.asarray(y).reshape(-1)
    y_idx = np.array([mapa[klasa] for klasa in y])
    
    n_klas = len(mapa)
    one_hot = np.zeros((len(y_idx), n_klas))
    one_hot[np.arange(len(y_idx)), y_idx] = 1.0
    
    return one_hot


def macro_f1_score(y_true, y_pred):
    y_true = np.asarray(y_true).reshape(-1)
    y_pred = np.asarray(y_pred).reshape(-1)

    klasy = np.unique(np.concatenate([y_true, y_pred]))
    f1_scores = []

    for klasa in klasy:
        tp = np.sum((y_true == klasa) & (y_pred == klasa))
        fp = np.sum((y_true != klasa) & (y_pred == klasa))
        fn = np.sum((y_true == klasa) & (y_pred != klasa))

        precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0

        if precision + recall == 0:
            f1 = 0.0
        else:
            f1 = 2 * precision * recall / (precision + recall)

        f1_scores.append(f1)

    return np.mean(f1_scores)

def accuracy_score_np(y_true, y_pred):
    y_true = np.asarray(y_true).reshape(-1)
    y_pred = np.asarray(y_pred).reshape(-1)
    return np.mean(y_true == y_pred)

## Klasa perceptronu, funkcja forward i backward

In [23]:
class MLP:

    def __init__(self, wielkosci_warstw, funkcja_akt, metoda, optimizer, beta=0.7, rho=0.9, eps=1e-8, seed=42, loss_target=None, task="regression", output_activation=None):

        self.wielkosci_warstw = wielkosci_warstw

        self.n_warstw = len(wielkosci_warstw) - 1

        self.optimizer = optimizer
        self.beta = beta
        self.rho = rho
        self.eps = eps
        self.seed = seed
        self.loss_target = loss_target

        self.task = task
        self.output_activation = output_activation
        self.output_activation_name = None

        # Inicjalizacja parametrów
    
        self.wagi, self.biasy = initiate_parameters(wielkosci_warstw, metoda, seed=self.seed)

        # zdefiniowanie funkcji aktywacji i ich pochodnych

        self.f_akt = []
        self.poch_f_akt = []

        for i in range(len(wielkosci_warstw) - 1):   
            if i < len(wielkosci_warstw) - 2:
                if funkcja_akt == sigmoid:
                    self.f_akt.append(sigmoid)
                    self.poch_f_akt.append(sigmoid_prime)
                elif funkcja_akt == relu:
                    self.f_akt.append(relu)
                    self.poch_f_akt.append(relu_prime)
                elif funkcja_akt == linear:
                    self.f_akt.append(linear)
                    self.poch_f_akt.append(linear_prime)
                elif funkcja_akt == tanh:
                    self.f_akt.append(tanh)
                    self.poch_f_akt.append(tanh_prime)
                elif funkcja_akt == leaky_relu:
                    self.f_akt.append(leaky_relu)
                    self.poch_f_akt.append(leaky_relu_prime)
                else:
                    raise ValueError("Nieobsługiwana funkcja aktywacji warstw ukrytych.")
            else:
                if self.output_activation is not None:
                    out_act = self.output_activation
                else:
                    if self.task == "classification":
                        out_act = softmax
                    else:
                        out_act = linear

                if out_act == sigmoid:
                    self.f_akt.append(sigmoid)
                    self.poch_f_akt.append(sigmoid_prime)
                    self.output_activation_name = "sigmoid"

                elif out_act == relu:
                    self.f_akt.append(relu)
                    self.poch_f_akt.append(relu_prime)
                    self.output_activation_name = "relu"

                elif out_act == linear:
                    self.f_akt.append(linear)
                    self.poch_f_akt.append(linear_prime)
                    self.output_activation_name = "linear"

                elif out_act == tanh:
                    self.f_akt.append(tanh)
                    self.poch_f_akt.append(tanh_prime)
                    self.output_activation_name = "tanh"

                elif out_act == leaky_relu:
                    self.f_akt.append(leaky_relu)
                    self.poch_f_akt.append(leaky_relu_prime)
                    self.output_activation_name = "leaky_relu"

                elif out_act == softmax:
                    self.f_akt.append(softmax)
                    self.poch_f_akt.append(None)   # nie używamy softmax_prime w tej ścieżce
                    self.output_activation_name = "softmax"

                else:
                    raise ValueError("Nieobsługiwana funkcja aktywacji warstwy wyjściowej.")

        # atrybuty potrzebne do algorytmu propagacji wstecznej

        self.cache_A = []
        self.cache_Z = []

    def _init_optimizer_state(self):
        self.vW = [np.zeros_like(W) for W in self.wagi]
        self.vb = [np.zeros_like(b) for b in self.biasy]

        self.sW = [np.zeros_like(W) for W in self.wagi]
        self.sb = [np.zeros_like(b) for b in self.biasy]

    def forward(self, X):

        # czyszczenie wartości funkcji aktywacji i preaaktywacji

        self.cache_A = []
        self.cache_Z = []

        # X to macierz z danymi wejściowymi o wymiarach (liczba_próbek, liczba cech)

        A = X
        self.cache_A.append(X)  # wyjście "zerowej" warstwy to po prostu dane wejściowe

        for i in range(self.n_warstw):
            W = self.wagi[i]
            b = self.biasy[i]
            g = self.f_akt[i]

            Z = A @ W + b
            self.cache_Z.append(Z)

            A = g(Z)
            self.cache_A.append(A)

        return A
    
    def backward(self, y_true):

        self.dW = [None] * self.n_warstw
        self.db = [None] * self.n_warstw
        
        y_pred = self.cache_A[-1]

        if self.task == "classification" and self.output_activation_name == "softmax":
            dZ = cross_entropy_grad_from_softmax(y_true, y_pred)
        else:
            dA = mse_grad(y_true, y_pred)
            dZ = dA * self.poch_f_akt[-1](self.cache_Z[-1])

        self.dW[-1] = self.cache_A[-2].T @ dZ
        self.db[-1] = np.sum(dZ, axis=0, keepdims=True)

        for i in reversed(range(self.n_warstw - 1)):
            dA_prev = dZ @ self.wagi[i+1].T
            dZ_prev = dA_prev * self.poch_f_akt[i](self.cache_Z[i])
            self.dW[i] = self.cache_A[i].T @ dZ_prev
            self.db[i] = np.sum(dZ_prev, axis=0, keepdims=True)
            dZ = dZ_prev

    def update_params(self, learning_rate):

        if self.optimizer == "sgd":
            for i in range(self.n_warstw):
                self.wagi[i] -= learning_rate * self.dW[i]
                self.biasy[i] -= learning_rate * self.db[i]

        elif self.optimizer == "momentum":
            for i in range(self.n_warstw):
                self.vW[i] = self.beta * self.vW[i] - learning_rate * self.dW[i]
                self.vb[i] = self.beta * self.vb[i] - learning_rate * self.db[i]

                self.wagi[i] += self.vW[i]
                self.biasy[i] += self.vb[i]

        elif self.optimizer == "rmsprop":
            for i in range(self.n_warstw):
                self.sW[i] = self.rho * self.sW[i] + (1.0 - self.rho) * (self.dW[i] ** 2)
                self.sb[i] = self.rho * self.sb[i] + (1.0 - self.rho) * (self.db[i] ** 2)

                self.wagi[i] -= learning_rate * self.dW[i] / (np.sqrt(self.sW[i]) + self.eps)
                self.biasy[i] -= learning_rate * self.db[i] / (np.sqrt(self.sb[i]) + self.eps)

        else:
            raise ValueError("Nieznany optimizer.")

    def compute_loss(self, y_true, y_pred):

        if self.task == "classification" and self.output_activation_name == "softmax":
            return cross_entropy_loss(y_true, y_pred)
        else:
            return mse_loss(y_true, y_pred)   
    

    def fit(self, X_train, y_train, learning_rate, n_epochs, batch_size, scaler_y=None, X_test=None, y_test=None, seed=None, verbose=True):

        if y_train.ndim == 1:
            y_train = y_train.reshape(-1, 1)

        if y_train.shape[1] != self.wielkosci_warstw[-1]:
            raise ValueError("Niezgodny wymiar targetu i warstwy wyjściowej.")

        if seed is None:
            seed = self.seed
        rng = np.random.default_rng(seed)

        self.historia_wag = []
        self.historia_biasow = []

        self.historia_strat = []
        self.historia_strat_oryg = []
        self.historia_test_strat = []
        self.historia_test_strat_oryg = []
        
        self.historia_accuracy = []
        self.historia_f_measure = []

        y_true = y_train

        self._init_optimizer_state()

        if batch_size <= 0:
            raise ValueError("Batch_size nie może być mniejszy/równy od 0")
        
        n = len(X_train)

        for i in range(n_epochs):

            if batch_size < len(X_train):
                indeksy = rng.permutation(n)
                X_epoch = X_train[indeksy]
                y_epoch = y_true[indeksy]
                n_batches = math.ceil(len(X_train)/batch_size)

            elif batch_size >= len(X_train):
                n_batches = 1
                X_epoch = X_train
                y_epoch = y_true

            for j in range(n_batches):
                start = j * batch_size
                end = (j + 1) * batch_size
                X_batch = X_epoch[start:end]
                y_batch = y_epoch[start:end]

                self.forward(X_batch)
                self.backward(y_batch)
                self.update_params(learning_rate)
                
            y_pred_train = self.forward(X_train)
            loss_train = self.compute_loss(y_train, y_pred_train)
            self.historia_strat.append(loss_train)

            if self.task == "regression" and scaler_y is not None:
                y_train_org = scaler_y.inverse_transform(y_train)
                y_pred_train_org = scaler_y.inverse_transform(y_pred_train)
                loss_train_org = mse_loss(y_train_org, y_pred_train_org)
                self.historia_strat_oryg.append(loss_train_org)

            if X_test is not None and y_test is not None:
                y_pred_test = self.predict(X_test)
                loss_test = self.compute_loss(y_test, y_pred_test)
                self.historia_test_strat.append(loss_test)

                if self.task == "regression" and scaler_y is not None:
                    y_test_org = scaler_y.inverse_transform(y_test)
                    y_pred_test_org = scaler_y.inverse_transform(y_pred_test)
                    self.historia_test_strat_oryg.append(mse_loss(y_test_org, y_pred_test_org))

            kopia_wag = [macierz.copy() for macierz in self.wagi]
            kopia_biasow = [macierz.copy() for macierz in self.biasy]
            self.historia_wag.append(kopia_wag)
            self.historia_biasow.append(kopia_biasow)

            if verbose and ((i + 1) % max(1, n_epochs // 10) == 0):
                print(f"Epoka {i + 1}/{n_epochs}, loss={loss_train:.6f}")

            if self.task == "regression" and self.loss_target is not None and len(self.historia_strat_oryg) > 0:
                if self.historia_strat_oryg[-1] < (self.loss_target / 10):
                    if verbose:
                        print(f"Epoka {i+1}: train_loss={self.historia_strat_oryg[-1]:.6f} < {self.loss_target} = loss_target")
                    break

            if self.task == "classification":
                y_true_train_labels = self._targets_to_labels(y_train)
                y_pred_train_labels = self.predict_classes(X_train)

                acc_train = accuracy_score_np(y_true_train_labels, y_pred_train_labels)
                f1_train = macro_f1_score(y_true_train_labels, y_pred_train_labels)

                self.historia_accuracy.append(acc_train)
                self.historia_f_measure.append(f1_train)
        

    def predict(self, X_test):
        return self.forward(X_test)
    
    def predict_classes(self, X_test):
        y_pred = self.forward(X_test)

        if self.output_activation_name == "sigmoid" and y_pred.shape[1] == 1:
            return (y_pred.ravel() >= 0.5).astype(int)

        return np.argmax(y_pred, axis=1)
    
    def _targets_to_labels(self, y):
        y = np.asarray(y)

        # one-hot, np. shape (n, K)
        if y.ndim == 2 and y.shape[1] > 1:
            return np.argmax(y, axis=1)

        # binarna klasyfikacja, np. shape (n, 1)
        if y.ndim == 2 and y.shape[1] == 1:
            return y.ravel().astype(int)

        # już podane jako etykiety klas
        return y.reshape(-1).astype(int)
    

## Funkcja do ładowania danych

In [24]:
def _wczytaj_csv_auto_index(path):

    df_probe = pd.read_csv(path)

    first_col_name = str(df_probe.columns[0]).strip().lower()
    first_col_values = df_probe.iloc[:, 0].values

    nazwa_wyglada_na_indeks = first_col_name in ["unnamed: 0", "index", ""]

    wartosci_wygladaja_na_indeks = False
    if len(first_col_values) > 0:
        try:
            wartosci_num = pd.to_numeric(first_col_values)
            wartosci_wygladaja_na_indeks = (wartosci_num == range(len(df_probe))).all()
        except Exception:
            wartosci_wygladaja_na_indeks = False

    if nazwa_wyglada_na_indeks or wartosci_wygladaja_na_indeks:
        return pd.read_csv(path, index_col=0)

    return df_probe

In [25]:
def zaladuj_dane(prefix):

    train_path = f"{prefix}-training.csv"
    test_path = f"{prefix}-test.csv"

    train_df = _wczytaj_csv_auto_index(train_path)
    test_df = _wczytaj_csv_auto_index(test_path)

    X_train_raw = train_df[['x']].values
    y_train_raw = train_df[['y']].values

    X_test = test_df[['x']].values
    y_test = test_df[['y']].values

    scaler_X = StandardScaler()
    X_train_scaled = scaler_X.fit_transform(X_train_raw)
    X_test_scaled = scaler_X.transform(X_test)

    scaler_y = StandardScaler()
    y_train_scaled = scaler_y.fit_transform(y_train_raw)
    y_test_scaled = scaler_y.transform(y_test)

    return (
        X_train_raw, y_train_raw,
        X_test, y_test,
        X_train_scaled, y_train_scaled,
        X_test_scaled, y_test_scaled,
        scaler_X, scaler_y
    )

## Funkcje do rysowania wykresów

In [26]:
def _upewnij_liste(obiekt):
    if isinstance(obiekt, (list, tuple)):
        return list(obiekt)
    return [obiekt]

def _sprawdz_modele_i_etykiety(modele, optymizery):
    modele = _upewnij_liste(modele)
    optymizery = _upewnij_liste(optymizery)

    if len(modele) != len(optymizery):
        raise ValueError("Liczba modeli musi być równa liczbie etykiet optymizerów.")

    return modele, optymizery


def _oblicz_predykcje_modeli(modele, X_model, scaler_y=None):
    y_preds = []

    for model in modele:
        y_pred = model.predict(X_model)

        if scaler_y is not None:
            y_pred = scaler_y.inverse_transform(y_pred)

        y_preds.append(y_pred)

    return y_preds


def _posortuj_po_x(X, y):
    X = np.asarray(X).reshape(-1)
    y = np.asarray(y).reshape(-1)

    idx = np.argsort(X)
    return X[idx], y[idx]


# 1. HISTORIA MSE NA TRAIN

def rysuj_historie_mse_train(modele, optymizery, dataset_name="", ax=None):
    modele, optymizery = _sprawdz_modele_i_etykiety(modele, optymizery)

    if ax is None:
        fig, ax = plt.subplots(figsize=(8, 5))

    for model, opt_name in zip(modele, optymizery):
        if not hasattr(model, "historia_strat_oryg"):
            raise AttributeError(f"Model '{opt_name}' nie ma atrybutu historia_strat_oryg.")

        historia = model.historia_strat_oryg
        ax.plot(range(1, len(historia) + 1), historia, label=opt_name)

    ax.set_yscale("log")
    ax.set_xlabel("Epoka")
    ax.set_ylabel("MSE treningowe")
    ax.set_title(f"Historia MSE (train) - {dataset_name}")
    ax.grid(True)
    ax.legend()

# 2. PORÓWNANIE WARTOŚCI PRAWDZIWYCH I PRZEWIDYWANYCH

def rysuj_porownanie_prawda_vs_pred(X_plot, y_true, y_preds, optymizery, title="", ax=None):
    y_preds = _upewnij_liste(y_preds)
    optymizery = _upewnij_liste(optymizery)

    if len(y_preds) != len(optymizery):
        raise ValueError("Liczba predykcji musi być równa liczbie etykiet optymizerów.")

    if ax is None:
        fig, ax = plt.subplots(figsize=(8, 5))

    X_sorted, y_true_sorted = _posortuj_po_x(X_plot, y_true)
    ax.scatter(X_sorted, y_true_sorted, label="wartości prawdziwe", linewidth=2, s=10)

    idx = np.argsort(np.asarray(X_plot).reshape(-1))

    for y_pred, opt_name in zip(y_preds, optymizery):
        y_pred = np.asarray(y_pred).reshape(-1)
        y_pred_sorted = y_pred[idx]
        ax.scatter(X_sorted, y_pred_sorted, label=opt_name, s=10)

    ax.set_xlabel("X")
    ax.set_ylabel("y")
    ax.set_title(title)
    ax.grid(True)
    ax.legend()


# 5. FUNKCJA GŁÓWNA - CAŁY PANEL WYKRESÓW

def rysuj_panel_porownawczy(
    modele,
    optymizery,
    X_train_plot,
    y_train,
    X_test_plot,
    y_test,
    X_train_model=None,
    X_test_model=None,
    scaler_y=None,
    dataset_name="",
    figsize=None
):

    modele, optymizery = _sprawdz_modele_i_etykiety(modele, optymizery)
    n_models = len(modele)

    if X_train_model is None:
        X_train_model = X_train_plot
    if X_test_model is None:
        X_test_model = X_test_plot

    y_preds_train = _oblicz_predykcje_modeli(modele, X_train_model, scaler_y=scaler_y)
    y_preds_test = _oblicz_predykcje_modeli(modele, X_test_model, scaler_y=scaler_y)

    total_cols = max(2, 2 * n_models)

    if figsize is None:
        figsize = (max(14, 4 * n_models), 16)

    fig = plt.figure(figsize=figsize, constrained_layout=True)
    gs = fig.add_gridspec(4, total_cols)

    # 1. wiersz - jeden szeroki wykres
    ax_train_mse = fig.add_subplot(gs[0, :])
    rysuj_historie_mse_train(
        modele=modele,
        optymizery=optymizery,
        dataset_name=dataset_name,
        ax=ax_train_mse
    )

    # 2. wiersz - train/test porównanie
    ax_train_pred = fig.add_subplot(gs[1, :total_cols // 2])
    ax_test_pred = fig.add_subplot(gs[1, total_cols // 2:])

    rysuj_porownanie_prawda_vs_pred(
        X_plot=X_train_plot,
        y_true=y_train,
        y_preds=y_preds_train,
        optymizery=optymizery,
        title=f"Train - {dataset_name}",
        ax=ax_train_pred
    )

    rysuj_porownanie_prawda_vs_pred(
        X_plot=X_test_plot,
        y_true=y_test,
        y_preds=y_preds_test,
        optymizery=optymizery,
        title=f"Test - {dataset_name}",
        ax=ax_test_pred
    )


    fig.suptitle(f"Porównanie modeli / optymizerów - {dataset_name}", fontsize=16)
    plt.show()

# NN4 - rozwiązywanie zadania klasyfikacji

## Loader danych do klasyfikacji

In [27]:
def zaladuj_dane_klasyfikacja(prefix, target_col=None):

    train_path = f"{prefix}-training.csv"
    test_path = f"{prefix}-test.csv"

    train_df = _wczytaj_csv_auto_index(train_path)
    test_df = _wczytaj_csv_auto_index(test_path)

    if target_col is None:
        target_col = train_df.columns[-1]

    X_train_raw = train_df.drop(columns=[target_col]).values
    y_train_raw = train_df[target_col].values

    X_test_raw = test_df.drop(columns=[target_col]).values
    y_test_raw = test_df[target_col].values

    scaler_X = StandardScaler()
    X_train_scaled = scaler_X.fit_transform(X_train_raw)
    X_test_scaled = scaler_X.transform(X_test_raw)

    y_train_one_hot, class_map = one_hot_encode(y_train_raw)
    y_test_one_hot = one_hot_encode_with_map(y_test_raw, class_map)

    return (
        X_train_raw, y_train_raw,
        X_test_raw, y_test_raw,
        X_train_scaled, y_train_one_hot,
        X_test_scaled, y_test_one_hot,
        scaler_X, class_map
    )

## Funkcja do rysowania wykresu

In [28]:
def rysuj_loss_vs_epoki(model_softmax, model_plain, dataset_name="dataset"):

    plt.figure(figsize=(10, 6))

    # train loss
    if len(model_softmax.historia_strat) > 0:
        plt.plot(model_softmax.historia_strat, label="Softmax - train loss")
    if len(model_plain.historia_strat) > 0:
        plt.plot(model_plain.historia_strat, label="Zwykła aktywacja - train loss")

    # test loss
    if len(model_softmax.historia_test_strat) > 0:
        plt.plot(model_softmax.historia_test_strat, linestyle="--", label="Softmax - test loss")
    if len(model_plain.historia_test_strat) > 0:
        plt.plot(model_plain.historia_test_strat, linestyle="--", label="Zwykła aktywacja - test loss")

    plt.title(f"Loss vs epoki - {dataset_name}")
    plt.xlabel("Epoka")
    plt.ylabel("Loss")
    plt.legend()
    plt.grid(True)
    plt.show()

In [29]:
def rysuj_granice_decyzyjne(
    model,
    X,
    y,
    scaler_X=None,
    dataset_name="dataset",
    model_name="model",
    grid_size=200,
    padding=0.5,
    show_points=True,
    alpha_bg=0.35
):

    X = np.asarray(X)
    y = np.asarray(y).reshape(-1)

    if X.ndim != 2 or X.shape[1] != 2:
        print(f"Nie rysuję granic decyzyjnych dla {dataset_name} ({model_name}), bo dane nie są 2D.")
        return

    # Zakres wykresu
    x_min, x_max = X[:, 0].min() - padding, X[:, 0].max() + padding
    y_min, y_max = X[:, 1].min() - padding, X[:, 1].max() + padding

    # Siatka o kontrolowanej gęstości
    xs = np.linspace(x_min, x_max, grid_size)
    ys = np.linspace(y_min, y_max, grid_size)
    xx, yy = np.meshgrid(xs, ys)

    grid = np.column_stack([xx.ravel(), yy.ravel()])

    # Skalowanie siatki, jeśli model był uczony na danych skalowanych
    if scaler_X is not None:
        grid_model = scaler_X.transform(grid)
    else:
        grid_model = grid

    # Jedna hurtowa predykcja
    Z = model.predict_classes(grid_model)
    Z = np.asarray(Z).reshape(xx.shape)

    plt.figure(figsize=(8, 6))
    plt.contourf(xx, yy, Z, levels=np.arange(Z.max() + 2) - 0.5, alpha=alpha_bg)

    if show_points:
        plt.scatter(X[:, 0], X[:, 1], c=y, edgecolors="k", s=20)

    plt.title(f"Granice decyzyjne - {dataset_name} - {model_name}")
    plt.xlabel("x1")
    plt.ylabel("x2")
    plt.grid(True)
    plt.show()

## Funkcja do porównania funkcji softmax z inną funkcją na wyjściu

In [30]:
def eksperyment_klasyfikacja(
    dataset_prefix,
    architektura,
    funkcja_akt_hidden=relu,
    metoda="he",
    optimizer="sgd",
    learning_rate=0.01,
    n_epochs=3000,
    batch_size=32,
    seed=42,
    output_activation_plain=sigmoid,
    target_col=None,
    verbose=True,
    rysuj_loss=True,
    rysuj_granice=True
):
    
    (
        X_train_raw, y_train_raw,
        X_test_raw, y_test_raw,
        X_train_scaled, y_train_one_hot,
        X_test_scaled, y_test_one_hot,
        scaler_X, class_map
    ) = zaladuj_dane_klasyfikacja(dataset_prefix, target_col=target_col)

    n_classes = y_train_one_hot.shape[1]

    if architektura[-1] != n_classes:
        raise ValueError(
            f"Ostatnia warstwa musi mieć tyle neuronów, ile jest klas. "
            f"Masz {architektura[-1]}, a klas jest {n_classes}."
        )

    # 1) model z softmax
    model_softmax = MLP(
        wielkosci_warstw=architektura,
        funkcja_akt=funkcja_akt_hidden,
        metoda=metoda,
        optimizer=optimizer,
        seed=seed,
        task="classification",
        output_activation=softmax
    )

    t0 = time.time()
    model_softmax.fit(
        X_train=X_train_scaled,
        y_train=y_train_one_hot,
        learning_rate=learning_rate,
        n_epochs=n_epochs,
        batch_size=batch_size,
        X_test=X_test_scaled,
        y_test=y_test_one_hot,
        verbose=verbose
    )
    t_softmax = time.time() - t0

    y_pred_softmax = model_softmax.predict_classes(X_test_scaled)
    f1_softmax = macro_f1_score(y_test_raw, y_pred_softmax)
    acc_softmax = accuracy_score_np(y_test_raw, y_pred_softmax)

    # 2) model ze zwykłą aktywacją na wyjściu
    model_plain = MLP(
        wielkosci_warstw=architektura,
        funkcja_akt=funkcja_akt_hidden,
        metoda=metoda,
        optimizer=optimizer,
        seed=seed,
        task="classification",
        output_activation=output_activation_plain
    )

    t0 = time.time()
    model_plain.fit(
        X_train=X_train_scaled,
        y_train=y_train_one_hot,
        learning_rate=learning_rate,
        n_epochs=n_epochs,
        batch_size=batch_size,
        X_test=X_test_scaled,
        y_test=y_test_one_hot,
        verbose=verbose
    )
    t_plain = time.time() - t0

    y_pred_plain = model_plain.predict_classes(X_test_scaled)
    f1_plain = macro_f1_score(y_test_raw, y_pred_plain)
    acc_plain = accuracy_score_np(y_test_raw, y_pred_plain)

    print(f"\nZbiór: {dataset_prefix}")
    print("=== Softmax + cross-entropy ===")
    print(f"Accuracy: {acc_softmax:.4f}")
    print(f"F-measure: {f1_softmax:.4f}")
    print(f"Czas treningu: {t_softmax:.2f} s")

    print("\n=== Zwykła aktywacja na wyjściu + MSE ===")
    print(f"Accuracy: {acc_plain:.4f}")
    print(f"F-measure: {f1_plain:.4f}")
    print(f"Czas treningu: {t_plain:.2f} s")

    # wykresy
    
    if rysuj_loss:
        rysuj_loss_vs_epoki(
            model_softmax=model_softmax,
            model_plain=model_plain,
            dataset_name=dataset_prefix
        )

    if rysuj_granice:

        rysuj_granice_decyzyjne(
        model=model_softmax,
        X=X_test_raw,
        y=y_test_raw,
        scaler_X=scaler_X,
        dataset_name=dataset_prefix,
        model_name="Softmax",
        grid_size=150
        )

        rysuj_granice_decyzyjne(
        model=model_plain,
        X=X_test_raw,
        y=y_test_raw,
        scaler_X=scaler_X,
        dataset_name=dataset_prefix,
        model_name="Zwykła aktywacja",
        grid_size=150
        )

    return {
        "softmax": {
            "model": model_softmax,
            "accuracy": acc_softmax,
            "f1": f1_softmax,
            "time": t_softmax
        },
        "plain": {
            "model": model_plain,
            "accuracy": acc_plain,
            "f1": f1_plain,
            "time": t_plain
        },
        "class_map": class_map
    }

# NN5 - Testowanie różnych funkcji aktywacji

Poprawiona wersja eksperymentów: średnie trajektorie MSE z 5 uruchomień, dopasowanie najlepszego modelu i tabele wyników.

In [31]:

# Poprawiona sekcja NN5:
# - dla każdego zbioru trenuje 5 modeli dla każdej metody/architektury,
# - rysuje tylko średnią trajektorię MSE i dopasowanie najlepszego modelu,
# - generuje tabelę metryk: mean/std/min/max MSE,
# - generuje tabelę architektura × funkcja aktywacji z uśrednionym MSE.

def _arch_label(architektura):
    return "-".join(map(str, architektura))


def _mean_curve(histories):
    """Zwraca średnią trajektorię po epokach. Ucina historie do wspólnej długości."""
    histories = [np.asarray(h, dtype=float).reshape(-1) for h in histories if len(h) > 0]
    if len(histories) == 0:
        return np.array([])
    min_len = min(len(h) for h in histories)
    return np.mean(np.vstack([h[:min_len] for h in histories]), axis=0)


def _mse_denorm(model, X_scaled, y_raw, scaler_y):
    y_pred_scaled = model.predict(X_scaled)
    y_pred = scaler_y.inverse_transform(y_pred_scaled)
    return float(np.mean((np.asarray(y_raw) - y_pred) ** 2))


def trenuj_serie_regresja_nn5(
    dataset_name,
    funkcja_akt,
    architektura,
    aktywacja_label,
    metoda="xavier",
    optimizer="rmsprop",
    learning_rate=0.001,
    n_epochs=1000,
    batch_size=32,
    seeds=(42, 43, 44, 45, 46),
    verbose=False
):
    """
    Trenuje kilka modeli o tej samej konfiguracji.
    Nie zmienia klasy MLP ani logiki fit(), tylko opakowuje eksperyment.
    """
    (
        X_train_raw, y_train_raw,
        X_test_raw, y_test_raw,
        X_train_scaled, y_train_scaled,
        X_test_scaled, y_test_scaled,
        scaler_X, scaler_y
    ) = zaladuj_dane(dataset_name)

    modele = []
    train_mse = []
    test_mse = []
    historie_train = []

    for seed in seeds:
        model = MLP(
            wielkosci_warstw=architektura,
            funkcja_akt=funkcja_akt,
            metoda=metoda,
            optimizer=optimizer,
            seed=seed,
            task="regression"
        )

        model.fit(
            X_train=X_train_scaled,
            y_train=y_train_scaled,
            learning_rate=learning_rate,
            n_epochs=n_epochs,
            batch_size=batch_size,
            scaler_y=scaler_y
        )

        mse_train = _mse_denorm(model, X_train_scaled, y_train_raw, scaler_y)
        mse_test = _mse_denorm(model, X_test_scaled, y_test_raw, scaler_y)

        modele.append(model)
        train_mse.append(mse_train)
        test_mse.append(mse_test)

        if hasattr(model, "historia_strat_oryg") and len(model.historia_strat_oryg) > 0:
            historie_train.append(model.historia_strat_oryg)
        elif hasattr(model, "historia_strat") and len(model.historia_strat) > 0:
            # awaryjnie, gdyby historia denormalizowana nie była dostępna
            # metryki końcowe nadal liczone są jawnie po denormalizacji.
            historie_train.append(model.historia_strat)

        if verbose:
            print(
                f"{dataset_name} | {aktywacja_label} | {_arch_label(architektura)} | "
                f"seed={seed} | train MSE={mse_train:.4f} | test MSE={mse_test:.4f}"
            )

    best_idx = int(np.argmin(test_mse))

    return {
        "dataset": dataset_name,
        "activation": aktywacja_label,
        "architecture": _arch_label(architektura),
        "architecture_raw": architektura,
        "models": modele,
        "best_model": modele[best_idx],
        "best_seed": list(seeds)[best_idx],
        "best_test_mse": float(test_mse[best_idx]),
        "train_mse_values": np.asarray(train_mse, dtype=float),
        "test_mse_values": np.asarray(test_mse, dtype=float),
        "mean_train_curve": _mean_curve(historie_train),
        "X_train_raw": X_train_raw,
        "y_train_raw": y_train_raw,
        "X_test_raw": X_test_raw,
        "y_test_raw": y_test_raw,
        "X_train_scaled": X_train_scaled,
        "X_test_scaled": X_test_scaled,
        "scaler_y": scaler_y
    }


def uruchom_eksperyment_nn5(
    datasets,
    konfiguracje,
    seeds=(42, 43, 44, 45, 46),
    metoda="xavier",
    optimizer="rmsprop",
    learning_rate=0.001,
    n_epochs=1000,
    batch_size=32,
    verbose=False
):
    """
    datasets: lista nazw zbiorów, np. ["multimodal-large", "steps-large"]
    konfiguracje: lista słowników:
        {
            "activation": "tanh",
            "func": tanh,
            "architectures": [[1, 8, 1], [1, 16, 1]]
        }
    """
    wyniki = []

    for dataset_name in datasets:
        for cfg in konfiguracje:
            for architektura in cfg["architectures"]:
                wynik = trenuj_serie_regresja_nn5(
                    dataset_name=dataset_name,
                    funkcja_akt=cfg["func"],
                    architektura=architektura,
                    aktywacja_label=cfg["activation"],
                    metoda=metoda,
                    optimizer=optimizer,
                    learning_rate=learning_rate,
                    n_epochs=n_epochs,
                    batch_size=batch_size,
                    seeds=seeds,
                    verbose=verbose
                )
                wyniki.append(wynik)

        rysuj_wykresy_dla_zbioru_nn5(dataset_name, wyniki)

    tabela_metryk = zbuduj_tabele_metryk_nn5(wyniki)
    tabela_heatmap = zbuduj_tabele_architektura_aktywacja_nn5(wyniki)

    display(tabela_metryk)
    pokaz_tabele_heatmap_nn5(tabela_heatmap)

    return wyniki, tabela_metryk, tabela_heatmap


def zbuduj_tabele_metryk_nn5(wyniki):
    rows = []

    for w in wyniki:
        test_vals = w["test_mse_values"]
        train_vals = w["train_mse_values"]

        rows.append({
            "zbiór": w["dataset"],
            "funkcja aktywacji": w["activation"],
            "architektura": w["architecture"],
            "średnie MSE train": np.mean(train_vals),
            "średnie MSE test": np.mean(test_vals),
            "std MSE test": np.std(test_vals, ddof=1),
            "min MSE test": np.min(test_vals),
            "max MSE test": np.max(test_vals),
            "najlepszy seed": w["best_seed"],
            "MSE test najlepszego modelu": w["best_test_mse"]
        })

    df = pd.DataFrame(rows)
    df = df.sort_values(["zbiór", "średnie MSE test", "funkcja aktywacji", "architektura"])
    return df.reset_index(drop=True)


def zbuduj_tabele_architektura_aktywacja_nn5(wyniki):
    rows = []

    for w in wyniki:
        rows.append({
            "zbiór": w["dataset"],
            "architektura": w["architecture"],
            "funkcja aktywacji": w["activation"],
            "średnie MSE test": float(np.mean(w["test_mse_values"]))
        })

    df_long = pd.DataFrame(rows)

    df_pivot = (
        df_long
        .pivot_table(
            index=["zbiór", "architektura"],
            columns="funkcja aktywacji",
            values="średnie MSE test",
            aggfunc="mean"
        )
        .sort_index()
    )

    return df_pivot


def pokaz_tabele_heatmap_nn5(df_pivot, precision=4, cmap="RdYlGn_r"):
    styled = (
        df_pivot.style
        .format(f"{{:.{precision}f}}")
        .background_gradient(cmap=cmap, axis=None)
        .set_caption("Porównanie MSE: architektury × funkcje aktywacji — wartości uśrednione z 5 modeli")
    )
    display(styled)
    return styled


def rysuj_srednie_trajektorie_mse_nn5(wyniki_zbioru, dataset_name, ax=None):
    if ax is None:
        fig, ax = plt.subplots(figsize=(12, 5))

    for w in wyniki_zbioru:
        curve = w["mean_train_curve"]
        if len(curve) == 0:
            continue

        label = f'{w["activation"]}, {w["architecture"]}'
        ax.plot(np.arange(1, len(curve) + 1), curve, label=label)

    ax.set_yscale("log")
    ax.set_xlabel("Epoka")
    ax.set_ylabel("Średnie MSE treningowe, denormalizowane")
    ax.set_title(f"Średnia trajektoria MSE z 5 modeli - {dataset_name}")
    ax.grid(True)
    ax.legend()


def rysuj_dopasowanie_najlepszych_nn5(wyniki_zbioru, dataset_name, split="test", ax=None):
    if ax is None:
        fig, ax = plt.subplots(figsize=(12, 5))

    pierwszy = wyniki_zbioru[0]

    if split == "train":
        X_raw = pierwszy["X_train_raw"]
        y_raw = pierwszy["y_train_raw"]
        X_scaled = pierwszy["X_train_scaled"]
        title = f"Dopasowanie najlepszych modeli - train - {dataset_name}"
    else:
        X_raw = pierwszy["X_test_raw"]
        y_raw = pierwszy["y_test_raw"]
        X_scaled = pierwszy["X_test_scaled"]
        title = f"Dopasowanie najlepszych modeli - test - {dataset_name}"

    x_flat = np.asarray(X_raw).reshape(-1)
    y_flat = np.asarray(y_raw).reshape(-1)
    idx = np.argsort(x_flat)

    ax.scatter(x_flat[idx], y_flat[idx], s=12, label="wartości prawdziwe")

    for w in wyniki_zbioru:
        y_pred_scaled = w["best_model"].predict(X_scaled)
        y_pred = w["scaler_y"].inverse_transform(y_pred_scaled).reshape(-1)

        label = f'{w["activation"]}, {w["architecture"]}'
        ax.scatter(x_flat[idx], y_pred[idx], s=12, label=label)

    ax.set_xlabel("X")
    ax.set_ylabel("y")
    ax.set_title(title)
    ax.grid(True)
    ax.legend()


def rysuj_wykresy_dla_zbioru_nn5(dataset_name, wyniki):
    wyniki_zbioru = [w for w in wyniki if w["dataset"] == dataset_name]

    fig = plt.figure(figsize=(16, 10), constrained_layout=True)
    gs = fig.add_gridspec(2, 2)

    ax_mse = fig.add_subplot(gs[0, :])
    rysuj_srednie_trajektorie_mse_nn5(wyniki_zbioru, dataset_name, ax=ax_mse)

    ax_train = fig.add_subplot(gs[1, 0])
    rysuj_dopasowanie_najlepszych_nn5(wyniki_zbioru, dataset_name, split="train", ax=ax_train)

    ax_test = fig.add_subplot(gs[1, 1])
    rysuj_dopasowanie_najlepszych_nn5(wyniki_zbioru, dataset_name, split="test", ax=ax_test)

    fig.suptitle(f"NN5 - porównanie funkcji aktywacji i architektur - {dataset_name}", fontsize=16)
    plt.show()


## Uruchomienie eksperymentów regresyjnych


In [ ]:

# Konfiguracja eksperymentów NN5.
# W razie potrzeby skróć listę zbiorów lub liczbę epok, jeśli notebook ma wykonywać się szybciej.

ZBIORY_NN5 = [
    "multimodal-large"
]

ARCHITEKTURY_NN5 = [
    [1, 8, 1],
    [1, 16, 1],
    [1, 32, 1],
    [1, 8, 8, 1],
    [1, 16, 16, 1],
    [1, 32, 32, 1],
    [1, 4, 8, 4, 1],
    [1, 8, 16, 8, 1],
    [1, 16, 32, 16, 1],
]

KONFIGURACJE_NN5 = [
    {"activation": "sigmoid", "func": sigmoid, "architectures": ARCHITEKTURY_NN5},
    {"activation": "linear",  "func": linear,  "architectures": ARCHITEKTURY_NN5},
    {"activation": "tanh",    "func": tanh,    "architectures": ARCHITEKTURY_NN5},
    {"activation": "relu",    "func": relu,    "architectures": ARCHITEKTURY_NN5},
]

wyniki_nn5, tabela_metryk_nn5, tabela_arch_akty_nn5 = uruchom_eksperyment_nn5(
    datasets=ZBIORY_NN5,
    konfiguracje=KONFIGURACJE_NN5,
    seeds=(42, 43, 44, 45, 46),
    metoda="xavier",
    optimizer="rmsprop",
    learning_rate=0.001,
    n_epochs=500,
    batch_size=32,
    verbose=True
)


Epoka 50/500, loss=0.381608
Epoka 100/500, loss=0.329836
Epoka 150/500, loss=0.287886
Epoka 200/500, loss=0.276758
Epoka 250/500, loss=0.266568
Epoka 300/500, loss=0.229188
Epoka 350/500, loss=0.131548
Epoka 400/500, loss=0.092208
Epoka 450/500, loss=0.075861
Epoka 500/500, loss=0.071337
multimodal-large | sigmoid | 1-8-1 | seed=42 | train MSE=369.3316 | test MSE=376.9910
Epoka 50/500, loss=0.382016
Epoka 100/500, loss=0.274699
Epoka 150/500, loss=0.169492
Epoka 200/500, loss=0.073905
Epoka 250/500, loss=0.064351
Epoka 300/500, loss=0.060408
Epoka 350/500, loss=0.058291
Epoka 400/500, loss=0.056790
Epoka 450/500, loss=0.055639
Epoka 500/500, loss=0.054488
multimodal-large | sigmoid | 1-8-1 | seed=43 | train MSE=282.0987 | test MSE=290.6857
Epoka 50/500, loss=0.391388
Epoka 100/500, loss=0.339392
Epoka 150/500, loss=0.324907
Epoka 200/500, loss=0.318560
Epoka 250/500, loss=0.312925
Epoka 300/500, loss=0.306797
Epoka 350/500, loss=0.227566
Epoka 400/500, loss=0.136966
Epoka 450/500, loss

## Funkcje do porównywania modeli w zadaniu klasyfikacji


In [ ]:
def porownaj_architektury_klasyfikacja(
    funkcja_akt_hidden,
    architektury,
    X_train_raw,
    y_train_raw,
    X_test_raw,
    y_test_raw,
    X_train_scaled,
    y_train_one_hot,
    X_test_scaled,
    y_test_one_hot,
    scaler_X=None,
    class_map=None,
    metoda="he",
    optimizer="rmsprop",
    learning_rate=0.001,
    n_epochs=3000,
    batch_size=32,
    seed=42,
    dataset_name="dataset",
    output_activation=softmax,
    verbose=True,
    rysuj=True
):
    modele = []
    etykiety = []
    acc_wyniki = []
    f1_wyniki = []

    n_classes = y_train_one_hot.shape[1]

    for i, architektura in enumerate(architektury):
        if architektura[-1] != n_classes:
            raise ValueError(
                f"Architektura {architektura} ma na wyjściu {architektura[-1]} neuronów, "
                f"a klas jest {n_classes}."
            )

        model = MLP(
            wielkosci_warstw=architektura,
            funkcja_akt=funkcja_akt_hidden,
            metoda=metoda,
            optimizer=optimizer,
            seed=seed + i,
            task="classification",
            output_activation=output_activation
        )

        model.fit(
            X_train=X_train_scaled,
            y_train=y_train_one_hot,
            learning_rate=learning_rate,
            n_epochs=n_epochs,
            batch_size=batch_size,
            X_test=X_test_scaled,
            y_test=y_test_one_hot,
            verbose=verbose
        )

        y_pred_test = model.predict_classes(X_test_scaled)
        acc = accuracy_score_np(y_test_raw, y_pred_test)
        f1 = macro_f1_score(y_test_raw, y_pred_test)

        modele.append(model)
        etykiety.append("-".join(map(str, architektura)))
        acc_wyniki.append(acc)
        f1_wyniki.append(f1)

        print(f"Architektura {etykiety[-1]} | accuracy={acc:.4f} | macro F1={f1:.4f}")

    if rysuj:
        rysuj_panel_porownawczy_klasyfikacja(
            modele=modele,
            etykiety=etykiety,
            X_train_raw=X_train_raw,
            y_train_raw=y_train_raw,
            X_test_raw=X_test_raw,
            y_test_raw=y_test_raw,
            X_train_scaled=X_train_scaled,
            X_test_scaled=X_test_scaled,
            scaler_X=scaler_X,
            dataset_name=dataset_name,
            acc_wyniki=acc_wyniki,
            f1_wyniki=f1_wyniki
        )

    return modele, etykiety, acc_wyniki, f1_wyniki

In [ ]:
def rysuj_historie_loss_klasyfikacja(modele, etykiety, dataset_name="", ax=None):
    modele = _upewnij_liste(modele)
    etykiety = _upewnij_liste(etykiety)

    if len(modele) != len(etykiety):
        raise ValueError("Liczba modeli musi być równa liczbie etykiet.")

    if ax is None:
        fig, ax = plt.subplots(figsize=(8, 5))

    for model, label in zip(modele, etykiety):
        historia = None

        if hasattr(model, "historia_strat"):
            historia = model.historia_strat
        elif hasattr(model, "historia_loss_train"):
            historia = model.historia_loss_train

        if historia is None or len(historia) == 0:
            print(f"Brak historii loss dla modelu {label}")
            continue

        ax.plot(range(1, len(historia) + 1), historia, label=label)

    ax.set_xlabel("Epoka")
    ax.set_ylabel("Loss treningowy")
    ax.set_title(f"Historia loss - {dataset_name}")
    ax.grid(True)
    ax.legend()

In [ ]:
def rysuj_panel_porownawczy_klasyfikacja(
    modele,
    etykiety,
    X_train_raw,
    y_train_raw,
    X_test_raw,
    y_test_raw,
    X_train_scaled,
    X_test_scaled,
    scaler_X=None,
    dataset_name="dataset",
    acc_wyniki=None,
    f1_wyniki=None
):
    modele = _upewnij_liste(modele)
    etykiety = _upewnij_liste(etykiety)

    if len(modele) != len(etykiety):
        raise ValueError("Liczba modeli musi być równa liczbie etykiet.")

    n_models = len(modele)

    fig = plt.figure(figsize=(max(14, 4 * n_models), 12), constrained_layout=True)
    gs = fig.add_gridspec(3, max(2, n_models))

    # 1. historia loss
    ax_loss = fig.add_subplot(gs[0, :])
    rysuj_historie_loss_klasyfikacja(
        modele=modele,
        etykiety=etykiety,
        dataset_name=dataset_name,
        ax=ax_loss
    )

    # 2. metryki
    if acc_wyniki is None or f1_wyniki is None:
        acc_wyniki = []
        f1_wyniki = []
        for model in modele:
            y_pred = model.predict_classes(X_test_scaled)
            acc_wyniki.append(accuracy_score_np(y_test_raw, y_pred))
            f1_wyniki.append(macro_f1_score(y_test_raw, y_pred))

    ax_metrics = fig.add_subplot(gs[1, :])
    rysuj_metryki_klasyfikacji(
        etykiety=etykiety,
        acc_wyniki=acc_wyniki,
        f1_wyniki=f1_wyniki,
        dataset_name=dataset_name,
        ax=ax_metrics
    )

    # 3. granice decyzyjne
    for i, (model, label) in enumerate(zip(modele, etykiety)):
        ax = fig.add_subplot(gs[2, i % max(2, n_models)])

        X = np.asarray(X_test_raw)
        y = np.asarray(y_test_raw).reshape(-1)

        if X.ndim != 2 or X.shape[1] != 2:
            ax.text(0.5, 0.5, "Brak granic decyzyjnych\n(dane nie są 2D)", ha="center", va="center")
            ax.set_title(label)
            ax.axis("off")
            continue

        x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
        y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5

        xs = np.linspace(x_min, x_max, 200)
        ys = np.linspace(y_min, y_max, 200)
        xx, yy = np.meshgrid(xs, ys)

        grid = np.column_stack([xx.ravel(), yy.ravel()])
        grid_model = scaler_X.transform(grid) if scaler_X is not None else grid

        Z = model.predict_classes(grid_model)
        Z = np.asarray(Z).reshape(xx.shape)

        ax.contourf(xx, yy, Z, levels=np.arange(Z.max() + 2) - 0.5, alpha=0.35)
        ax.scatter(X[:, 0], X[:, 1], c=y, edgecolors="k", s=20)
        ax.set_title(label)
        ax.set_xlabel("x1")
        ax.set_ylabel("x2")
        ax.grid(True)

    fig.suptitle(f"Porównanie architektur - klasyfikacja - {dataset_name}", fontsize=16)
    plt.show()

In [ ]:
def porownaj_architektury_klasyfikacja_dla_zbioru(
    dataset_name,
    funkcja_akt_hidden,
    architektury,
    metoda="he",
    optimizer="rmsprop",
    learning_rate=0.01,
    n_epochs=3000,
    batch_size=32,
    seed=42,
    output_activation=softmax,
    target_col=None,
    verbose=True,
    rysuj=True
):
    (
        X_train_raw, y_train_raw,
        X_test_raw, y_test_raw,
        X_train_scaled, y_train_one_hot,
        X_test_scaled, y_test_one_hot,
        scaler_X, class_map
    ) = zaladuj_dane_klasyfikacja(dataset_name, target_col=target_col)

    return porownaj_architektury_klasyfikacja(
        funkcja_akt_hidden=funkcja_akt_hidden,
        architektury=architektury,
        X_train_raw=X_train_raw,
        y_train_raw=y_train_raw,
        X_test_raw=X_test_raw,
        y_test_raw=y_test_raw,
        X_train_scaled=X_train_scaled,
        y_train_one_hot=y_train_one_hot,
        X_test_scaled=X_test_scaled,
        y_test_one_hot=y_test_one_hot,
        scaler_X=scaler_X,
        class_map=class_map,
        metoda=metoda,
        optimizer=optimizer,
        learning_rate=learning_rate,
        n_epochs=n_epochs,
        batch_size=batch_size,
        seed=seed,
        dataset_name=dataset_name,
        output_activation=output_activation,
        verbose=verbose,
        rysuj=rysuj
    )

## rings5-regular

In [ ]:
architektury = [
    [2, 8, 16, 8, 5],
    [2, 16, 32, 16, 5],
]

modele, etykiety, acc_wyniki, f1_wyniki = porownaj_architektury_klasyfikacja_dla_zbioru(
    dataset_name="rings5-regular",
    funkcja_akt_hidden=tanh,
    architektury=architektury,
    metoda="xavier",
    optimizer="rmsprop",
    learning_rate=0.001,
    n_epochs=3000,
    batch_size=16,
    seed=42,
    output_activation=softmax,
    verbose=False,
    rysuj=True
)

## rings3-regular

In [ ]:
architektury = [
    [2, 8, 16, 8, 3],
    [2, 16, 32, 16, 3],
]

modele, etykiety, acc_wyniki, f1_wyniki = porownaj_architektury_klasyfikacja_dla_zbioru(
    dataset_name="rings3-regular",
    funkcja_akt_hidden=tanh,
    architektury=architektury,
    metoda="xavier",
    optimizer="rmsprop",
    learning_rate=0.001,
    n_epochs=3000,
    batch_size=16,
    seed=42,
    output_activation=softmax,
    verbose=False,
    rysuj=True
)